In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [3]:
df = pd.read_json(
    "Sarcasm_Headlines_Dataset_v2.json",
    lines=True
)

df.head()

,is_sarcastic,headline,article_link
0,1,thirtysomething scientists unveil doomsday clo...,https://www.theonion.com/thirtysomething-scien...
1,0,dem rep. totally nails why congress is falling...,https://www.huffingtonpost.com/entry/donna-edw...
2,0,eat your veggies: 9 deliciously different recipes,https://www.huffingtonpost.com/entry/eat-your-...
3,1,inclement weather prevents liar from getting t...,https://local.theonion.com/inclement-weather-p...
4,1,mother comes pretty close to using word 'strea...,https://www.theonion.com/mother-comes-pretty-c...


In [4]:
df = df[
    ["headline", "is_sarcastic"]
]

df.columns = [
    "text",
    "label"
]

df.head()

,text,label
0,thirtysomething scientists unveil doomsday clo...,1
1,dem rep. totally nails why congress is falling...,0
2,eat your veggies: 9 deliciously different recipes,0
3,inclement weather prevents liar from getting t...,1
4,mother comes pretty close to using word 'strea...,1


In [5]:
df.isnull().sum()

,0
text,0
label,0


In [6]:
df = df.dropna()

df = df.drop_duplicates()

df.shape

(28503, 2)

In [7]:
X = df["text"]

y = df["label"]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [9]:
tokenizer = Tokenizer(
    num_words=5000
)

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)

X_test_seq = tokenizer.texts_to_sequences(X_test)

In [10]:
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=50
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=50
)

In [11]:
model = Sequential([

    Embedding(
        5000,
        32
    ),

    LSTM(32),

    Dense(
        1,
        activation="sigmoid"
    )

])

In [12]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [13]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/5
571/571 ━━━━━━━━━━━━━━━━━━━━ 19s 29ms/step - accuracy: 0.7690 - loss: 0.4766 - val_accuracy: 0.8496 - val_loss: 0.3509
Epoch 2/5
571/571 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - accuracy: 0.8807 - loss: 0.2839 - val_accuracy: 0.8516 - val_loss: 0.3414
Epoch 3/5
571/571 ━━━━━━━━━━━━━━━━━━━━ 22s 25ms/step - accuracy: 0.9114 - loss: 0.2227 - val_accuracy: 0.8459 - val_loss: 0.3655
Epoch 4/5
571/571 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.9307 - loss: 0.1795 - val_accuracy: 0.8439 - val_loss: 0.3912
Epoch 5/5
571/571 ━━━━━━━━━━━━━━━━━━━━ 20s 23ms/step - accuracy: 0.9463 - loss: 0.1421 - val_accuracy: 0.8310 - val_loss: 0.4493


In [14]:
test_loss, test_accuracy = model.evaluate(
    X_test_pad,
    y_test
)

test_accuracy

179/179 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8458 - loss: 0.4242


0.8458165526390076

The forget gate allows an LSTM to decide which old information should be kept and which information should be discarded. This helps it remember useful information over longer sequences better than a plain RNN.